<style>
.jp-RenderedHTMLCommon h1 { color:#ff9900; font-size:2.3em; }
.jp-RenderedHTMLCommon h2 { color:#2563a8; }
.jp-RenderedHTMLCommon blockquote { border-left:6px solid #ff9900; background:#fff7e8; padding:.6em 1em; }
.jp-RenderedHTMLCommon table { font-size:.92em; }
</style>

# AWS Glue: Introduction
## Serverless data integration, metadata, discovery, and ETL

**Theory lesson · direct teaching · no hands-on activities**

> By the end, you should be able to explain how crawlers, the Data Catalog, databases, tables, jobs, and orchestration fit together—and why Region choice matters.


# Learning outcomes

After this lesson, you can:

- distinguish ETL, ELT, and data integration;
- describe AWS Glue's major components and their boundaries;
- trace metadata from a crawler into Catalog databases and tables;
- explain how a Glue job reads, transforms, and writes data;
- reason about Regions, IAM, networking, security, orchestration, and cost;
- choose when Glue is—and is not—the right service.


# The data integration problem

Organizations collect data in object stores, operational databases, warehouses, streams, and SaaS systems. The difficult work is rarely just moving bytes. A reliable pipeline must also:

- discover structure and partitions;
- reconcile inconsistent schemas and types;
- clean, validate, enrich, and join data;
- record metadata so people and services can find it;
- schedule, retry, monitor, and secure every run.

**Data integration** is the larger discipline. ETL is one important pattern inside it.


# ETL and ELT

| Pattern | Flow | Where transformation occurs | Typical reason |
|---|---|---|---|
| **ETL** | Extract → Transform → Load | Before the final target | Curated output, data quality, controlled schemas |
| **ELT** | Extract → Load → Transform | In the destination engine | Powerful warehouse/lakehouse compute, retain raw data |

Neither is universally superior. Modern architectures often use both: land immutable raw data first, then run transformation jobs to create trusted layers.

> Glue is broader than “ETL jobs”: it provides discovery, cataloging, authoring, execution, and orchestration capabilities.


# What is AWS Glue?

AWS Glue is a **serverless data integration service**. You define metadata and transformation intent; AWS operates the underlying service infrastructure.

Core ideas:

- **discover** data with crawlers and classifiers;
- **organize** technical metadata in the Glue Data Catalog;
- **author** transformations visually or with code;
- **execute** batch or streaming jobs using managed engines;
- **orchestrate** dependencies with triggers and workflows;
- **monitor** runs through Glue and integrated AWS observability services.

Serverless does not mean “no configuration”: you still own data design, permissions, code, capacity choices, and cost control.


# One mental model

![Conceptual AWS Glue flow](assets/glue_concept_architecture.png)

**Sources → discovery → shared metadata → transformation → destinations**

The Catalog describes data; it does not normally contain the business data itself. Jobs use those descriptions to locate and interpret data.


# Glue at a glance

![AWS Glue big-picture slide](assets/source_slides/slide_asset_08.jpg)

The source slide emphasizes three planes:

1. **Metadata plane** — Data Catalog, databases, tables, partitions.
2. **Development plane** — job authoring, generated scripts, notebooks/interactive tooling.
3. **Execution plane** — managed compute that runs transformations and writes results.

Keeping these planes separate prevents a common misconception: creating a Catalog table does **not** run an ETL job.


# Core components

| Component | Primary responsibility | Produces or manages |
|---|---|---|
| Data Catalog | Central metadata repository | Databases, tables, partitions, schema properties |
| Crawler | Inspects data stores | Catalog table/partition updates |
| Classifier | Interprets a data format/schema | Classification and inferred schema |
| Connection | Stores connection/network metadata | Reusable access configuration |
| Job | Executes transformation logic | Curated data and run history |
| Trigger | Starts jobs/crawlers | Scheduled, on-demand, or conditional actions |
| Workflow | Groups and tracks a pipeline | Dependency-level run view |
| Data quality capability | Evaluates rules | Quality results and observations |


# The Glue Data Catalog

The Data Catalog is a persistent technical metadata store shared by Glue and several analytics services. It answers questions such as:

- Where is the dataset?
- What format is it in?
- What columns and types are expected?
- How is it partitioned?
- Which serializer/deserializer or table properties are needed?

It can describe data in object storage and compatible external stores. Think of it as a **library catalog**: the catalog card describes and locates the book; it is not the book.


# Catalog hierarchy

```text
AWS account + Region
└── Glue Data Catalog
    ├── Database: logical namespace
    │   ├── Table: schema + location + format properties
    │   │   ├── Columns
    │   │   └── Partitions (optional)
    │   └── Table ...
    └── Database ...
```

- A **database** groups related table definitions; it is not a database server.
- A **table** is metadata that points to the underlying data.
- A **partition** describes a subset, often mapped to path keys such as `year=2026/month=09/`.


# Region scope: a crucial boundary

The Glue Data Catalog and Glue resources are **regional**. In practice:

- changing the AWS Console Region changes the databases, tables, crawlers, and jobs you see;
- identical names can exist independently in different Regions;
- infrastructure code, SDK/CLI calls, monitoring, and governance must target the intended Region;
- data may be physically stored elsewhere, but cross-Region access adds architecture, policy, latency, and cost considerations.

> Teach the identity of a Glue resource as **account + Region + resource name/ARN**, not name alone.


# Glue databases

A Glue database is a namespace for Catalog tables. Good database boundaries often reflect:

- environment: `dev`, `test`, `prod`;
- data layer: `raw`, `curated`, `serving`;
- business domain: `sales`, `finance`, `operations`;
- ownership or governance boundary.

Avoid treating the database name as the only security control. Authorization is enforced through IAM and, where adopted, centralized data-lake governance controls.


# Glue tables

A Catalog table commonly records:

- database and table name;
- data location (for example, an object-store prefix);
- input/output format and serialization information;
- column names, types, and comments;
- partition keys and individual partitions;
- classification and custom key-value properties.

**Important:** deleting or changing table metadata is logically separate from deleting the underlying files. Likewise, changing a schema definition does not rewrite historical data.


# Crawlers: automated discovery

![Crawler overview from source slides](assets/source_slides/slide_asset_13.jpg)

A crawler connects to one or more data stores, selects a classifier, infers structure, and updates the Catalog. It can run on demand or on a schedule.

Typical flow:

1. Assume an IAM role.
2. Enumerate permitted data paths or objects.
3. Classify the format and infer schema.
4. Group compatible data into table definitions.
5. add/update tables and partitions according to crawler configuration.


# Classifiers and schema inference

![Crawler classifiers from source slides](assets/source_slides/slide_asset_14.jpg)

Classifiers help determine a dataset's format and schema. Built-in classifiers cover common structured and semi-structured formats; custom classifiers address organization-specific patterns.

Inference is a **proposal**, not business truth. Sampling, malformed files, mixed schemas, or ambiguous values can lead to surprising types. Production teams should validate inferred schemas and control schema evolution deliberately.


# Crawler strengths and limits

**Use crawlers when** datasets arrive or partitions change and automated discovery reduces operational work.

**Be cautious when**:

- folders contain unrelated schemas;
- type inference must be deterministic;
- frequent scans are wasteful;
- schema changes must pass review before publication;
- naming/grouping heuristics do not match the data model.

Alternatives include defining Catalog objects through infrastructure as code, SDK/API calls, or compatible engines. A mature platform may combine explicit tables with crawler-managed partitions.


# Glue jobs

A job is the executable unit of data processing. It combines:

- transformation code or a visual job graph;
- an IAM execution role;
- engine/runtime and language settings;
- worker/capacity configuration;
- arguments, libraries, connections, and temporary locations;
- retry, timeout, concurrency, and observability settings.

Common job styles include distributed Spark jobs, lighter Python-shell tasks, and streaming transformations. Available engines and versions evolve, so runtime selection should be verified for the target Region and account.


# What happens during a job run?

```text
Trigger / API / schedule
        ↓
Glue creates a job run and provisions managed capacity
        ↓
Execution role accesses sources, Catalog, secrets, and destinations
        ↓
Code reads → transforms → validates → writes
        ↓
Metrics, logs, status, and output artifacts are emitted
        ↓
Capacity is released; metadata/state may be updated
```

The **job definition** is reusable configuration. A **job run** is one execution with its own ID, arguments, status, and logs.


# Authoring choices

![Job authoring choices from source slides](assets/source_slides/slide_asset_15.jpg)

- **Visual authoring**: compose sources, transforms, and targets as a graph; useful for readability and standard patterns.
- **Generated script**: begin with service-generated code, then refine it.
- **Existing script**: bring tested code under your own development lifecycle.
- **Notebook/interactive development**: explore and debug transformations before packaging a repeatable job.

Production quality still needs version control, tests, deployment automation, and code review—regardless of the authoring surface.


# DynamicFrames and DataFrames

Glue's Spark-oriented tooling commonly exposes **DynamicFrames**, an abstraction designed for messy or evolving data, alongside standard Spark **DataFrames**.

| DynamicFrame emphasis | DataFrame emphasis |
|---|---|
| Flexible records and schema ambiguity | Explicit tabular schema |
| Glue-specific transforms and Catalog integration | Broad Spark SQL ecosystem and optimization |
| Helpful for semi-structured ingestion | Helpful for complex analytical transformations |

Pipelines may convert between them. Choose based on transformation needs, not habit.


# Handling schema change

Schema evolution can mean added columns, removed columns, renamed fields, type changes, or changing nested structures. Treat each differently.

Design questions:

- Is the change backward compatible?
- Should old and new records coexist?
- Will the crawler update metadata automatically?
- Can downstream engines interpret the new schema?
- Should the pipeline quarantine unexpected records?
- Who approves contract-breaking changes?

Glue can accommodate flexible data, but governance remains a human and organizational responsibility.


# Partitions and performance

Partitioning divides a dataset into subsets, commonly encoded in object paths. Engines can then skip irrelevant data through **partition pruning**.

Good partition keys are frequently filtered, reasonably distributed, and not excessively high-cardinality. Poor design creates either huge scans or too many tiny partitions/files.

The Catalog must know partitions before many query engines can prune them. Crawlers, explicit partition registration, and newer partition-management patterns are different ways to maintain that knowledge.


# Connections and networking

A Glue connection captures reusable connectivity metadata—for example, how a job reaches a JDBC source. Network reachability is separate from credentials and authorization.

For private resources, reason about:

- VPC, subnet, route tables, security groups, DNS;
- endpoints or controlled egress for required AWS services;
- credentials stored securely rather than embedded in scripts;
- the execution role's permission to retrieve secrets and access data.

An IAM policy can permit an action while the network still blocks the connection—and vice versa.


# Security: three questions

1. **Who may configure Glue?** The human/automation principal creating jobs, tables, roles, and workflows.
2. **What may the job do?** The execution role assumed by the managed runtime.
3. **What protects the data?** Resource policies, storage policies, encryption keys, network controls, and data-governance permissions.

Apply least privilege, separate development and production roles, encrypt data in transit and at rest, restrict log exposure, and audit control-plane activity. Sensitive row/column access may require governance beyond basic Catalog permissions.


# Orchestration

![Job composition and triggers from source slides](assets/source_slides/slide_asset_22.jpg)

- **Triggers** start jobs or crawlers on demand, on a schedule, or after conditions are met.
- **Workflows** group related jobs, crawlers, and triggers and track their dependency-level execution.
- **External orchestration** may be preferable when a pipeline spans many AWS services, accounts, or non-AWS systems.

Design for idempotency: rerunning the same logical input should not silently duplicate or corrupt results.


# Reliability and observability

A production pipeline should expose:

- job-run state, duration, retries, and capacity consumption;
- driver/executor logs and actionable error context;
- input/output counts, rejected records, and data-quality results;
- freshness and completeness of expected partitions;
- alerts tied to business impact, not only infrastructure failure.

Common failure families: permission denied, network timeout, schema mismatch, missing dependency, insufficient capacity, skew, out-of-memory, and bad input data.


# Cost model: reason in drivers

Avoid memorizing prices from old slides. Pricing and supported features can change. Instead, understand the drivers:

- execution capacity × run duration;
- crawler processing time and scan frequency;
- interactive/development session usage;
- Data Catalog request/storage levels beyond included allowances;
- related services: object storage requests, logs, network transfer, encryption keys, orchestration, and query engines.

Control cost by right-sizing, pruning input, using efficient columnar formats, compacting small files, stopping idle sessions, and avoiding unnecessary crawls. Verify current pricing before estimates.


# When Glue fits well

- Data lakes centered on AWS storage and analytics services.
- Managed distributed transformations without operating clusters.
- Shared metadata needed by multiple compatible engines.
- Batch pipelines with changing partitions and semi-structured data.
- Teams that value service integration over infrastructure control.

# When to evaluate alternatives

- Millisecond event processing or request/response workloads.
- Tiny transformations where distributed startup dominates.
- Highly specialized engines or deep cluster-level control.
- Cross-cloud portability as the primary requirement.


# End-to-end example: orders pipeline

1. Applications land daily order files in a **raw** object-store prefix.
2. A crawler discovers new partitions and updates `raw_sales.orders`.
3. A scheduled job reads the Catalog table, validates types, removes duplicates, standardizes currency, and joins reference data.
4. Bad records go to a quarantine path with failure reasons.
5. Curated columnar files are written partitioned by business date.
6. The curated Catalog table/partitions are updated.
7. Query and BI services read the curated dataset.
8. Monitoring checks job success **and** data freshness/quality.

Notice the separation: files hold data; Catalog tables describe it; jobs transform it; orchestration coordinates it.


# Common misconceptions

| Misconception | Correct model |
|---|---|
| “The Catalog stores my dataset.” | It primarily stores metadata that points to data. |
| “A crawler cleans data.” | It discovers/classifies metadata; a job performs transformation. |
| “A database is an RDS-like engine.” | A Glue database is a Catalog namespace. |
| “Serverless means zero operations.” | AWS runs infrastructure; you own design, access, reliability, and cost. |
| “Inferred schema is always correct.” | It is based on observed data and classifier behavior. |
| “A successful job guarantees good data.” | Technical success and data quality are different outcomes. |


# Architecture checklist

Before approving a Glue design, ask:

- Where are source and target data, and in which Regions?
- Who owns schema contracts and evolution?
- Are Catalog objects crawler-managed or explicitly defined?
- What is the partition and file-size strategy?
- Which role runs the job, and what is its least-privilege boundary?
- Does the runtime need private networking, secrets, or extra libraries?
- How are retries, bookmarks/checkpoints, idempotency, and backfills handled?
- What proves freshness, completeness, and correctness?
- Which usage dimensions dominate cost?


# Recap

**AWS Glue connects five ideas:**

1. **Discover** structure with crawlers and classifiers.
2. **Describe** datasets in regional Catalog databases, tables, and partitions.
3. **Transform** data with managed jobs.
4. **Coordinate** execution with triggers, workflows, or external orchestration.
5. **Govern and observe** the pipeline through permissions, networking, logs, metrics, and quality controls.

> The shortest accurate mental model: **Glue is a regional, serverless data-integration service whose metadata and compute components work together but remain distinct.**


# Knowledge check

1. Why is a Glue table not the same thing as the data it describes?
2. What changes when you switch the AWS Console Region?
3. When might explicit schema management be safer than crawler inference?
4. Which permissions belong to the job execution role?
5. Why can a technically successful run still be a failed data product?
6. When would you choose a DataFrame over a DynamicFrame?
7. Which design choices reduce both scan time and cost?

**Next notebook:** hands-on creation and execution of Glue resources.
